# Cora → centroid hypergraph (Bai et al.)

Citation graph from `dissertation/cora/` (`cora.content`, `cora.cites`) is turned into a **hypergraph** following the construction used alongside **hypergraph convolution** in:

> Song Bai, Feihu Zhang, Philip H. S. Torr, *Hypergraph Convolution and Hypergraph Attention*, arXiv:1901.08150 (2019). https://arxiv.org/abs/1901.08150

**Construction (as in that line of work):**

1. Vertices = all papers (articles).
2. For each paper $v$ as **centroid**, one hyperedge contains $v$ and every paper that has a **citation link** with $v$ (either $v$ cites $u$ or $u$ cites $v$). Equivalently: hyperedge $e_v = \{v\} \cup N(v)$ in the **undirected** graph obtained by ignoring edge direction.

Node indices `0 … n-1` follow the **row order** in `cora.content` (same as bag-of-words features and labels).

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
from hypergraphx import Hypergraph

cora_dir = Path("..") / "cora"
if not (cora_dir / "cora.content").exists():
    raise FileNotFoundError(cora_dir.resolve())

paper_ids = []
labels_raw = []
with open(cora_dir / "cora.content", encoding="utf-8", errors="replace") as f:
    for line in f:
        parts = line.strip().split("\t")
        paper_ids.append(parts[0])
        labels_raw.append(parts[-1])

n = len(paper_ids)
pid_to_idx = {p: i for i, p in enumerate(paper_ids)}

neigh = defaultdict(set)
with open(cora_dir / "cora.cites", encoding="utf-8", errors="replace") as f:
    for line in f:
        cited, citing = line.strip().split()
        if cited in pid_to_idx and citing in pid_to_idx:
            u, v = pid_to_idx[cited], pid_to_idx[citing]
            neigh[u].add(v)
            neigh[v].add(u)

edge_list = []
for i in range(n):
    nodes = {i} | neigh[i]
    edge_list.append(tuple(sorted(nodes)))

H = Hypergraph(edge_list)

classes = sorted(set(labels_raw))
cls_to_y = {c: j for j, c in enumerate(classes)}
y = np.array([cls_to_y[c] for c in labels_raw], dtype=int)

print(f"Papers (nodes): {n}")
print(f"Centroid hyperedges: {len(edge_list)} (one per paper)")
print(f"Hypergraphx: |V|={len(H.get_nodes())}, |E|={len(H.get_edges())}")
print(f"Classes: {classes}")